
# Laboratory 5D — Does a more advanced electronic-structure model improve the result?

The main Laboratory 5 calculation used PBE, a generalized-gradient
approximation (GGA). We now compare it with an instructor-provided B3LYP
reference dataset.

B3LYP is a **hybrid functional**: it includes a fraction of exact exchange and
is computationally more demanding than PBE.

The scientific question is not:

> *Is B3LYP automatically better because it is more expensive?*

Instead, we will ask:

> **Which predicted properties improve, which change only slightly, and which
> discrepancies remain?**

The same quasi-harmonic analysis strategy will be used for both datasets:
BM3 equation-of-state minimization at ambient pressure from 0 to 1500 K.



## 1. Inspect the B3LYP reference before running it

The B3LYP data are already assembled in:

```text
reference-results/mgo_b3lyp.yaml
```

The file contains the static energy–volume series and the volume-dependent
phonon frequencies required by Quantas.

Before comparing the results, inspect a few pieces of metadata.


In [ ]:

from pathlib import Path
import numpy as np
import yaml
import pandas as pd

with open("reference-results/mgo_b3lyp.yaml") as handle:
    b3_input = yaml.safe_load(handle)

volume = np.asarray(b3_input["volume"], dtype=float)
reference_index = int(b3_input["structure"]["reference_index"])
reference_volume = volume[reference_index]

pd.DataFrame(
    {
        "quantity": [
            "number of volume points",
            "q-points stored in QHA dataset",
            "reference primitive volume (A^3)",
            "minimum V/Vref",
            "maximum V/Vref",
        ],
        "B3LYP dataset": [
            len(volume),
            b3_input["qpoints"],
            reference_volume,
            volume.min() / reference_volume,
            volume.max() / reference_volume,
        ],
    }
)



Now compare this information with the current PBE dataset that you inspected in
Notebook 01.

### Question 1 — Is this a perfectly controlled functional benchmark?

Are the two datasets identical in every numerical detail apart from the
exchange–correlation functional?

Identify at least one numerical difference between the PBE and B3LYP reference
datasets and explain why the comparison should therefore be described as a
**model comparison** rather than a perfectly isolated one-variable test.

**Your answer:**  
_Write 3–4 lines here._



## 2. Run the B3LYP QHA with the same thermodynamic settings

Use the same BM3 minimization, temperature range and 10 K spacing used for the
main PBE calculation.


In [ ]:

!quantas qha run reference-results/mgo_b3lyp.yaml \
    --scheme freq \
    --minimization eos \
    --eos BM3 \
    --thermal-expansion mixed_derivative \
    --temperature 0 1500 10 \
    --pressure 0 0 1 \
    --energy-degree 3 \
    --frequency-degree 3 \
    --output mgo_qha_b3lyp_bm3_0GPa.hdf5 \
    --report mgo_qha_b3lyp_bm3_0GPa.log \
    --no-progress --force



The PBE result from Notebook 02 should already exist as:

```text
mgo_qha_bm3_0GPa.hdf5
```

We will use:

```python
PBE = "mgo_qha_bm3_0GPa.hdf5"
B3LYP = "mgo_qha_b3lyp_bm3_0GPa.hdf5"
```


In [ ]:

from mineral_science_tools.qha_lab import (
    compare_qha_models_table,
    plot_model_thermochemistry,
    plot_model_volume,
    plot_model_alpha,
    plot_model_bulk_modulus,
)

PBE = "mgo_qha_bm3_0GPa.hdf5"
B3LYP = "mgo_qha_b3lyp_bm3_0GPa.hdf5"

thermo = pd.read_csv("data/mgo_nist_janaf_thermochemistry.csv")
volume_exp = pd.read_csv("data/mgo_reeber1995_volume_expansion.csv")
bulk_exp = pd.read_csv("data/mgo_isaak1989_bulk_moduli.csv")

volume_exp = volume_exp[volume_exp["T (K)"] <= 1500]
bulk_exp = bulk_exp[bulk_exp["T (K)"] <= 1500]



## 3. Thermodynamic properties

### Entropy


In [ ]:

plot_model_thermochemistry(PBE, B3LYP, thermo, "S");



### Constant-pressure heat capacity


In [ ]:

plot_model_thermochemistry(PBE, B3LYP, thermo, "Cp");



### Enthalpy increment


In [ ]:

plot_model_thermochemistry(PBE, B3LYP, thermo, "H");



### Question 2 — Which thermodynamic quantities care about the functional?

Compare PBE and B3LYP for $S(T)$, $C_P(T)$ and
$H(T)-H(298.15\,\mathrm K)$.

Which curves change appreciably and which are relatively insensitive to the
electronic-structure model?

Does B3LYP improve the agreement with the thermochemical reference for every
property?

**Your answer:**  
_Write 5–6 lines here._



## 4. Absolute equilibrium volume


In [ ]:

plot_model_volume(
    PBE,
    B3LYP,
    volume_exp,
    normalized=False,
);



### Question 3 — The absolute structure

Which electronic-structure model gives the better absolute equilibrium volume?

Is the main difference between the two theoretical curves approximately a
constant offset, or does their temperature dependence also differ strongly?

**Your answer:**  
_Write 3–4 lines here._



## 5. Normalized thermal expansion of the volume


In [ ]:

plot_model_volume(
    PBE,
    B3LYP,
    volume_exp,
    normalized=True,
    reference_temperature=300.0,
);



### Question 4 — Absolute accuracy versus physical trend

After dividing each curve by its own 300 K volume, how different are PBE and
B3LYP?

Use this comparison to distinguish between:

- predicting the **absolute equilibrium structure**;
- predicting the **relative thermal response** of that structure.

**Your answer:**  
_Write 4–5 lines here._



## 6. Volumetric thermal expansion


In [ ]:

plot_model_alpha(
    PBE,
    B3LYP,
    volume_exp,
);



### Question 5

Does the model that gives the better absolute volume also give a clearly better
$\alpha_V(T)$?

Why is this comparison useful when deciding whether a more expensive
electronic-structure method is scientifically justified?

**Your answer:**  
_Write 3–4 lines here._



## 7. Isothermal bulk modulus


In [ ]:

plot_model_bulk_modulus(
    PBE,
    B3LYP,
    bulk_exp,
    "KT",
);



## 8. Adiabatic bulk modulus


In [ ]:

plot_model_bulk_modulus(
    PBE,
    B3LYP,
    bulk_exp,
    "KS",
);



### Question 6 — Structure and stiffness

Compare the calculated $K_T(T)$ and $K_S(T)$ with their matched experimental
references.

Does improving the equilibrium structure also improve the predicted elastic
response?

Discuss both the absolute modulus and its temperature dependence.

**Your answer:**  
_Write 5–6 lines here._



## 9. Put some numbers beside the plots

The figures show the trends. The table below gives a compact numerical
comparison at three temperatures.


In [ ]:

compare_qha_models_table(
    PBE,
    B3LYP,
    temperatures=(300.0, 1000.0, 1500.0),
)



### Question 7 — Is the additional computational cost buying useful accuracy?

B3LYP is higher than PBE on Jacob's ladder of density-functional
approximations, but that does not guarantee that every observable becomes more
accurate.

Based on **your plots and table**, answer:

1. Which properties improve clearly with B3LYP?
2. Which properties are already described adequately by PBE?
3. Which remaining discrepancies are unlikely to be solved simply by changing
   the exchange–correlation functional?
4. For this MgO problem, where would the additional computational cost of a
   hybrid functional be scientifically worthwhile?

**Your answer:**  
_Write 7–10 lines here._



## Take-home message

A more sophisticated electronic-structure model should be judged by what it
changes in the **observable predictions**, not by its position on a hierarchy
alone.

The useful comparison is therefore:

```text
electronic-structure approximation
        ↓
equilibrium structure + phonons
        ↓
quasi-harmonic free energy
        ↓
observable thermodynamic / thermoelastic properties
        ↓
comparison with experiment
```

The experiment decides whether the additional computational cost produces a
scientifically meaningful improvement.
